In [2]:
import pandas as pd
import numpy as np

# Create mapping from CRESCENDDI drugname to DrugBank ID

In [3]:
def map_drug_names_to_ids(mapping_path, drugbank_emb_path, write_output=False):
    """Map drug names from the mapping file to DrugBank IDs using the embedding description file.

    Args:
        mapping_path (str): Path to the Excel file containing drug mappings.
        drugbank_emb_path (str): Path to the CSV file containing DrugBank embeddings.
        write_output (bool, optional): Whether to write the merged output to an Excel file. Defaults to False.

    Returns:
        dict: A dictionary mapping drug concept names to DrugBank IDs.
    """
    drug_mappings = pd.read_excel(mapping_path)
    drug_mappings = drug_mappings[drug_mappings["RXNORM_CODE/RXNORM_EXTENSION_CODE (OHDSI)"] != 0]
    drug_mappings["DRUG_CONCEPT_NAME"] = drug_mappings["DRUG_CONCEPT_NAME"].str.lower()

    emb_desc = pd.read_csv(drugbank_emb_path, sep="\t")
    emb_desc = emb_desc[["Drug Name", "Drug ID"]]
    emb_desc["Drug Name"] = emb_desc["Drug Name"].str.lower()

    # 1. Merge using DRUG_CONCEPT_NAME
    merged_mappings = drug_mappings.merge(
        emb_desc[["Drug Name", "Drug ID"]], left_on="DRUG_CONCEPT_NAME", right_on="Drug Name", how="left"
    )

    # 2. Handle unmatched rows by trying DRUG_INITIAL_NAME
    unmatched = merged_mappings[merged_mappings["Drug ID"].isna()].copy()
    unmatched.drop(columns=["Drug Name", "Drug ID"], inplace=True)
    unmatched["DRUG_INITIAL_NAME"] = unmatched["DRUG_INITIAL_NAME"].str.lower()
    second_merge = unmatched.merge(
        emb_desc[["Drug Name", "Drug ID"]], left_on="DRUG_INITIAL_NAME", right_on="Drug Name", how="left"
    )

    # 3. Combine results
    final_merged = pd.concat([merged_mappings[~merged_mappings["Drug ID"].isna()], second_merge], ignore_index=True)

    print("Number of unmapped drugs:", final_merged["Drug ID"].isna().sum(), "out of", len(final_merged))

    if write_output:
        with pd.ExcelWriter("auxillary_datasets/drug_mappings_and_emb_desc.xlsx") as writer:
            final_merged.to_excel(writer, sheet_name="MergedMappings", index=False)
            emb_desc.to_excel(writer, sheet_name="EmbDesc", index=False)

    return final_merged.set_index("DRUG_CONCEPT_NAME")["Drug ID"].to_dict()

In [4]:
mapping_path = "/data/giobbi/CRESCENDDI/Data Record 4 - Drug mappings.xlsx"
drugbank_emb_path = "/data/giobbi/embeddings/DESC_GPT.csv"

name_to_id = map_drug_names_to_ids(mapping_path, drugbank_emb_path, write_output=False)

Number of unmapped drugs: 596 out of 5914


# Create CRESCENDDI Graph

In [12]:
pos_df = pd.read_excel("/data/giobbi/CRESCENDDI/Data Record 1 - Positive Controls.xlsx")
pos_df

,DRUG_1_CONCEPT_NAME,DRUG_2_CONCEPT_NAME,EVENT_CONCEPT_NAME,MDR_CODE,EVENT_SOURCE,BNF_SEV_LEVEL,ANSM_SEV_LEVEL,MICROMEDEX_SEV_LEVEL,BNF_EVID_LEVEL,MICROMEDEX_EVID_LEVEL
0,Bendroflumethiazide,desmopressin,Hyponatraemia,10021036,BNF+Micromedex,NaN,Take into consideration,Major,NaN,Theoretical
1,Chlorthalidone,desmopressin,Hyponatraemia,10021036,BNF+Micromedex,NaN,Take into consideration,Major,NaN,Theoretical
2,desmopressin,Clopamide,Hyponatraemia,10021036,BNF+Micromedex,NaN,Take into consideration,Major,NaN,Theoretical
3,Escitalopram,desmopressin,Hyponatraemia,10021036,BNF+Micromedex,NaN,Take into consideration,Major,NaN,Theoretical
4,Paroxetine,desmopressin,Hyponatraemia,10021036,BNF+Micromedex,NaN,Take into consideration,Major,NaN,Theoretical
...,...,...,...,...,...,...,...,...,...,...
10281,Cimetidine,Chloroquine,Cardiac arrest,10007515,Micromedex,Moderate,Take into consideration,Major,Study,Probable
10282,Cimetidine,Moclobemide,Tachypnoea,10043089,Micromedex,Mild,Precautions for use,Moderate,Study,Theoretical
10283,Cimetidine,Moclobemide,Anxiety,10002855,Micromedex,Mild,Precautions for use,Moderate,Study,Theoretical
10284,Cimetidine,Moclobemide,Tachycardia,10043071,Micromedex,Mild,Precautions for use,Moderate,Study,Theoretical


In [7]:
neg_df = pd.read_excel("/data/giobbi/CRESCENDDI/Data Record 2 - Negative Controls.xlsx")
neg_df


,DRUG_1_CONCEPT_NAME,DRUG_2_CONCEPT_NAME,EVENT_CONCEPT_NAME,MDR_CODE
0,Valproate,Isosorbide,Drug-induced liver injury,10072268
1,Bendroflumethiazide,Acyclovir,Thrombocytopenia,10043554
2,Midodrine,Amikacin,Blood creatinine increased,10005483
3,Amiodarone,quinapril,Deafness,10011878
4,Timolol,sunitinib,Tremor,10044565
...,...,...,...,...
4539,fluticasone,Clonidine,Neutropenia,10029354
4540,fluticasone,Perindopril,Dyskinesia,10013916
4541,bortezomib,Ceftriaxone,Muscular weakness,10028372
4542,olanzapine,meloxicam,Toxicity to various agents,10070863


In [8]:
def create_graph_from_edges(name_to_id):
    # Load positive and negative control edges
    pos_df = pd.read_excel("/data/giobbi/CRESCENDDI/Data Record 1 - Positive Controls.xlsx")
    pos_df["DRUG_1_CONCEPT_NAME"] = pos_df["DRUG_1_CONCEPT_NAME"].str.lower()
    pos_df["DRUG_2_CONCEPT_NAME"] = pos_df["DRUG_2_CONCEPT_NAME"].str.lower()
    pos_df = pos_df[["DRUG_1_CONCEPT_NAME", "DRUG_2_CONCEPT_NAME"]]
    pos_df.loc[:, "label"] = 1

    neg_df = pd.read_excel("/data/giobbi/CRESCENDDI/Data Record 2 - Negative Controls.xlsx")
    neg_df["DRUG_1_CONCEPT_NAME"] = neg_df["DRUG_1_CONCEPT_NAME"].str.lower()
    neg_df["DRUG_2_CONCEPT_NAME"] = neg_df["DRUG_2_CONCEPT_NAME"].str.lower()
    neg_df = neg_df[["DRUG_1_CONCEPT_NAME", "DRUG_2_CONCEPT_NAME"]]
    neg_df.loc[:, "label"] = 0

    edges = pd.concat([pos_df, neg_df], ignore_index=True)

    edges["Drug1"] = edges["DRUG_1_CONCEPT_NAME"].map(name_to_id)
    edges["Drug2"] = edges["DRUG_2_CONCEPT_NAME"].map(name_to_id)

    print(f"Dropping {edges['Drug1'].isna().sum() + edges['Drug2'].isna().sum()} edges with unmapped drugs out of {len(edges)} total edges.")
    edges = edges.dropna(subset=["Drug1", "Drug2"]).copy()

    # Remove duplicate edges considering undirected pairs (A-B is same as B-A)
    # Sort the drug IDs row-wise to normalize the order
    sorted_edges = np.sort(edges[["Drug1", "Drug2"]].values, axis=1)
    sorted_edges_df = pd.DataFrame(sorted_edges, columns=["Drug1", "Drug2"], index=edges.index)

    print(f"Dropping {sorted_edges_df.duplicated().sum()} duplicate edges out of {len(edges)} total edges.")
    edges = edges[~sorted_edges_df.duplicated()]

    return edges


In [9]:
edges = create_graph_from_edges(name_to_id)

Dropping 414 edges with unmapped drugs out of 14830 total edges.
Dropping 5625 duplicate edges out of 14416 total edges.


In [10]:
edges

,DRUG_1_CONCEPT_NAME,DRUG_2_CONCEPT_NAME,label,Drug1,Drug2
0,bendroflumethiazide,desmopressin,1,DB00436,DB00035
1,chlorthalidone,desmopressin,1,DB00310,DB00035
2,desmopressin,clopamide,1,DB00035,DB13792
3,escitalopram,desmopressin,1,DB01175,DB00035
4,paroxetine,desmopressin,1,DB00715,DB00035
...,...,...,...,...,...
14824,paroxetine,fosinopril,0,DB00715,DB00492
14825,fluticasone,clonidine,0,DB13867,DB00575
14826,fluticasone,perindopril,0,DB13867,DB00790
14827,bortezomib,ceftriaxone,0,DB00188,DB01212


In [11]:
# Write graph to file

#GRAPH_URL = "/data/giobbi/CRESCENDDI/CRESCENDDI.csv"
#edges[["Drug1", "Drug2", "label"]].to_csv(GRAPH_URL, index=False, sep="\t")

#pos_edges = edges[edges["label"] == 1]
#GRAPH_URL = "/data/giobbi/CRESCENDDI/positive_edges_CRESCENDDI.csv"
#pos_edges[["Drug1", "Drug2"]].to_csv(GRAPH_URL, index=False, sep="\t")